In [ ]:
# === Common notebook preamble (load llm_math package) ===
import sys
from pathlib import Path

# Candidate paths to locate the llm_math package
_candidates = [
    '.', 'src', '..', '../src',
    '/content/llm-math-book/src',
    '/content/llm-math-book',
    '/workspace/src',
    '/workspace',
]
# Also add parent directories of the current directory (when running from notebooks/)
for p in Path.cwd().parents:
    _candidates.append(str(p / 'src'))
    _candidates.append(str(p))

for p in _candidates:
    if p and p not in sys.path and Path(p).exists():
        sys.path.insert(0, p)

# Try importing llm_math
try:
    from llm_math import viz, bench, data
    _LLM_MATH_OK = True
except ImportError as e:
    _LLM_MATH_OK = False
    print(f"[Warning] Failed to load the llm_math package: {e}")
    print("  Clone the GitHub repository and run colab_setup.sh.")
# === End preamble ===


# Ch 01. Vectors and Spaces - The Starting Point of Linear Algebra

> **Learning Goals**
> - Understand vectors both as points in space and as arrows.
> - Verify the mathematical meaning of dot products, norms, and distance in code.
> - Understand why cosine similarity is central to LLM embeddings.

## 1.1 What Is a Vector?

A vector is an **ordered list of numbers**. It can be understood from two perspectives at the same time:

1. **Algebraic view**: $\mathbf{x} = [x_1, x_2, \ldots, x_n]^\top \in \mathbb{R}^n$
2. **Geometric view**: a point in space, or an arrow from the origin to that point

In LLMs, word embeddings are represented as 768-dimensional or 4096-dimensional vectors. We will first build intuition in 2 or 3 dimensions.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from llm_math.viz import plot_vector_2d, setup_axes_2d

# 2D vector example
a = np.array([3.0, 1.0])
b = np.array([-1.0, 2.0])
c = a + b   # vector addition

fig, ax = plt.subplots(1, 1, figsize=(7, 7))
plot_vector_2d(ax, a, color='blue', label='a = [3, 1]')
plot_vector_2d(ax, b, color='green', label='b = [-1, 2]')
plot_vector_2d(ax, c, color='red', label='a + b')
setup_axes_2d(ax, xlim=(-3, 6), ylim=(-3, 5))
ax.legend(fontsize=11)
ax.set_title('Vector addition: a + b', fontsize=13)
plt.tight_layout()
plt.savefig('../figures/ch01_vector_addition.png', dpi=100, bbox_inches='tight')
plt.show()
print("vector a =", a, "| b =", b, "| a + b =", c)


## 1.2 Vector Operations and Geometric Meaning

### Scalar Multiplication

$\alpha \mathbf{x} = [\alpha x_1, \alpha x_2, \ldots]^\top$

Scalar multiplication stretches or shrinks the length of a vector by $\alpha$. If $\alpha < 0$, the direction is reversed.

### Dot Product

$$\mathbf{a} \cdot \mathbf{b} = \sum_{i=1}^{n} a_i b_i = \|\mathbf{a}\| \|\mathbf{b}\| \cos\theta$$

The dot product measures **how strongly two vectors point in the same direction**.
- $\cos\theta = 1$ (same direction): maximum dot product
- $\cos\theta = 0$ (orthogonal): dot product = 0
- $\cos\theta = -1$ (opposite direction): minimum dot product


In [ ]:
# Dot product calculation
a = np.array([3.0, 1.0])
b = np.array([-1.0, 2.0])

dot = np.dot(a, b)
print(f"a · b = {a[0]}*{b[0]} + {a[1]}*{b[1]} = {dot}")

# Verify dot product = |a||b|cos(theta)
norm_a = np.linalg.norm(a)
norm_b = np.linalg.norm(b)
cos_theta = dot / (norm_a * norm_b)
theta_rad = np.arccos(np.clip(cos_theta, -1, 1))
theta_deg = np.degrees(theta_rad)
print(f"|a| = {norm_a:.4f}, |b| = {norm_b:.4f}")
print(f"cos(theta) = {cos_theta:.4f}")
print(f"theta = {theta_deg:.2f} degrees")
print(f"Validation: |a||b|cos(theta) = {norm_a * norm_b * cos_theta:.4f} == dot = {dot:.4f}")


## 1.3 Types of Norms

A norm is a function that measures the "length" of a vector. Several norms are commonly used:

$$\|\mathbf{x}\|_p = \left(\sum_{i=1}^{n} |x_i|^p\right)^{1/p}$$

- $L^1$ norm ($p=1$): $\sum_i |x_i|$ - Manhattan distance
- $L^2$ norm ($p=2$): $\sqrt{\sum_i x_i^2}$ - Euclidean distance
- $L^\infty$ norm: $\max_i |x_i|$ - maximum absolute value

In LLMs, the $L^2$ norm appears frequently in LayerNorm, weight initialization, and gradient clipping.


In [ ]:
# Compare different norms
x = np.array([3.0, -4.0])

l1 = np.sum(np.abs(x))
l2 = np.sqrt(np.sum(x**2))
linf = np.max(np.abs(x))

print(f"x = {x}")
print(f"L1 norm   = |3| + |-4|          = {l1}")
print(f"L2 norm   = sqrt(3^2 + 4^2)     = {l2}")
print(f"L-inf norm = max(|3|, |-4|)      = {linf}")
print()
print("NumPy Validation:")
print(f"  np.linalg.norm(x, 1)  = {np.linalg.norm(x, 1)}")
print(f"  np.linalg.norm(x, 2)  = {np.linalg.norm(x, 2)}")
print(f"  np.linalg.norm(x, np.inf) = {np.linalg.norm(x, np.inf)}")


## 1.4 Distance and Similarity

### Euclidean Distance

$$d(\mathbf{a}, \mathbf{b}) = \|\mathbf{a} - \mathbf{b}\|_2$$

### Cosine Similarity

$$\cos(\mathbf{a}, \mathbf{b}) = \frac{\mathbf{a} \cdot \mathbf{b}}{\|\mathbf{a}\| \|\mathbf{b}\|}$$

In LLM embeddings, cosine similarity is the standard way to measure word or sentence similarity.
Reasons:
1. The **direction** of an embedding vector often captures meaning, while its **magnitude** may reflect frequency or other factors.
2. In high dimensions, comparing only direction is often more robust than comparing raw distance.


In [ ]:
# Implement cosine similarity directly
def cosine_similarity(a, b):
    """Cosine Similarity (cosine similarity)."""
    dot = np.dot(a, b)
    norm_a = np.linalg.norm(a)
    norm_b = np.linalg.norm(b)
    if norm_a == 0 or norm_b == 0:
        return 0.0
    return dot / (norm_a * norm_b)

# Example: three toy word embeddings
king   = np.array([0.9, 0.8, 0.3])
queen  = np.array([0.85, 0.85, 0.4])
apple  = np.array([0.1, 0.2, 0.9])

print(f"cos(king, queen)  = {cosine_similarity(king, queen):.4f}  (should be high)")
print(f"cos(king, apple)  = {cosine_similarity(king, apple):.4f}  (should be low)")
print(f"cos(queen, apple) = {cosine_similarity(queen, apple):.4f} (should be low)")

# Compare Euclidean distances
print()
print(f"euclidean(king, queen)  = {np.linalg.norm(king - queen):.4f}  (should be small)")
print(f"euclidean(king, apple)  = {np.linalg.norm(king - apple):.4f}  (should be large)")


## 1.5 Vector Spaces and Subspaces

A **vector space** is a set of vectors that is closed under vector addition and scalar multiplication.

**Linear combination**: $\alpha \mathbf{a} + \beta \mathbf{b}$
- The result of multiplying vectors by scalars and adding them

**Span**: the set of all linear combinations that can be formed from given vectors
- $\text{span}(\{\mathbf{e}_1, \mathbf{e}_2\}) = \mathbb{R}^2$

**Basis**: a linearly independent set of vectors that spans a vector space
- Standard basis of $\mathbb{R}^n$: $\mathbf{e}_1 = [1,0,\ldots], \mathbf{e}_2 = [0,1,0,\ldots], \ldots$


In [ ]:
# Span visualization: the region generated by linear combinations of two vectors
e1 = np.array([1.0, 0.0])
e2 = np.array([0.0, 1.0])

# Multiple alpha and beta combinations
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Standard basis
ax = axes[0]
alphas = np.linspace(-2, 2, 20)
betas = np.linspace(-2, 2, 20)
points = []
for a in alphas:
    for b in betas:
        points.append(a * e1 + b * e2)
points = np.array(points)
ax.scatter(points[:, 0], points[:, 1], s=8, alpha=0.4, color='blue')
plot_vector_2d(ax, e1, color='red', label='e1')
plot_vector_2d(ax, e2, color='green', label='e2')
setup_axes_2d(ax, xlim=(-3, 3), ylim=(-3, 3))
ax.set_title('span({e1, e2}) = R^2 (standard basis)')
ax.legend()

# Two linearly dependent vectors (same direction)
ax = axes[1]
v1 = np.array([1.0, 1.0])
v2 = np.array([2.0, 2.0])  # same direction as v1
points = []
for a in alphas:
    for b in betas:
        points.append(a * v1 + b * v2)
points = np.array(points)
ax.scatter(points[:, 0], points[:, 1], s=8, alpha=0.4, color='blue')
plot_vector_2d(ax, v1, color='red', label='v1=[1,1]')
plot_vector_2d(ax, v2, color='green', label='v2=[2,2]')
setup_axes_2d(ax, xlim=(-3, 6), ylim=(-3, 6))
ax.set_title('span({v1, v2}) = line (linearly dependent!)')
ax.legend()
plt.tight_layout()
plt.savefig('../figures/ch01_span.png', dpi=100, bbox_inches='tight')
plt.show()
print("Left: linearly independent basis vectors span the entire 2D plane")
print("Right: linearly dependent vectors span only a line (1D subspace)")


## 1.6 Application: Word Embeddings and Cosine Similarity

In LLMs, words are represented as vectors with hundreds to thousands of dimensions. Words with similar meanings tend to point in similar directions.
Here we compute similarity with a simplified 3-dimensional embedding example.


In [ ]:
# Mini word embedding example (simplified to 3D)
# Assume each dimension represents [royalty, femininity, food-relatedness]
words = {
    'king':    np.array([0.99, 0.70, 0.05]),
    'queen':   np.array([0.95, 0.95, 0.05]),
    'man':     np.array([0.50, 0.10, 0.05]),
    'woman':   np.array([0.50, 0.90, 0.05]),
    'apple':   np.array([0.05, 0.40, 0.95]),
    'orange':  np.array([0.05, 0.30, 0.92]),
    'banana':  np.array([0.05, 0.20, 0.90]),
}

# Cosine similarity matrix computation
word_list = list(words.keys())
n = len(word_list)
sim_matrix = np.zeros((n, n))
for i, w1 in enumerate(word_list):
    for j, w2 in enumerate(word_list):
        sim_matrix[i, j] = cosine_similarity(words[w1], words[w2])

print("Cosine similarity matrix:")
print(f"{'':>10}", end='')
for w in word_list:
    print(f"{w:>10}", end='')
print()
for i, w1 in enumerate(word_list):
    print(f"{w1:>10}", end='')
    for j, w2 in enumerate(word_list):
        print(f"{sim_matrix[i, j]:>10.3f}", end='')
    print()

# Find the most similar word
print("\nWords most similar to 'king':")
king_idx = word_list.index('king')
sims = sim_matrix[king_idx].copy()
sims[king_idx] = -1  # Exclude itself
top2 = np.argsort(-sims)[:3]
for idx in top2:
    print(f"  {word_list[idx]:>10}: {sims[idx]:.4f}")


In [ ]:
# Visualize the similarity matrix as a heatmap
fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(sim_matrix, cmap='YlOrRd', vmin=0, vmax=1)
ax.set_xticks(range(n))
ax.set_yticks(range(n))
ax.set_xticklabels(word_list, rotation=45, ha='right')
ax.set_yticklabels(word_list)
plt.colorbar(im, ax=ax, label='Cosine Similarity')
ax.set_title('Word Embedding Cosine Similarity Heatmap')

# Show values
for i in range(n):
    for j in range(n):
        ax.text(j, i, f'{sim_matrix[i, j]:.2f}', ha='center', va='center',
                color='black' if sim_matrix[i, j] < 0.7 else 'white', fontsize=9)
plt.tight_layout()
plt.savefig('../figures/ch01_word_similarity.png', dpi=100, bbox_inches='tight')
plt.show()


## 1.7 Key Takeaways

| Concept | Formula | Meaning |
|---|---|---|
| Vector norm | $\|\mathbf{x}\|_p$ | Length of a vector |
| Dot product | $\mathbf{a} \cdot \mathbf{b}$ | Degree of shared direction |
| Cosine similarity | $\frac{\mathbf{a} \cdot \mathbf{b}}{\|\mathbf{a}\|\|\mathbf{b}\|}$ | Compare direction only, ignoring magnitude |
| Euclidean distance | $\|\mathbf{a} - \mathbf{b}\|_2$ | Straight-line distance |
| Span | $\text{span}(\{\mathbf{v}_i\})$ | All linear combinations that can be generated |

### Practice Problems
1. For 3D vectors $\mathbf{a} = [1, 2, 3]$ and $\mathbf{b} = [4, -1, 0]$, compute the dot product, $L^2$ norm, and cosine similarity.
2. What is the dot product when two vectors are orthogonal? Create an example where the answer is 0.
3. Are $\mathbf{v}_1 = [1, 2]$ and $\mathbf{v}_2 = [2, 4]$ linearly independent? Explain.
4. Numerically verify that $\|\mathbf{x}\|_1 \geq \|\mathbf{x}\|_2 \geq \|\mathbf{x}\|_\infty$ for an arbitrary 5D vector $\mathbf{x}$.
5. In the word embedding example, explain from the embedding-dimension perspective why the cosine similarity between 'man' and 'woman' is higher than that between 'man' and 'apple'.
